# Streaming x Estrutura das Séries

Pergunta: a entrada do streaming reduziu o número de temporadas e/ou episódios por temporada das séries? Isso se correlaciona com a qualidade (`averageRating`)?


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy.stats import mannwhitneyu, spearmanr

sns.set_theme(style="whitegrid")

RNG = np.random.default_rng(42)

STREAMING_YEAR = 2010


In [ ]:
episodes = pd.read_parquet('db/episodes.parquet')
ratings = pd.read_parquet('db/ratings.parquet')

series_basics = pd.read_csv(
    'db/title.basics.tsv',
    sep='\t',
    na_values='\\N',
    usecols=['tconst', 'titleType', 'primaryTitle', 'startYear']
)

series_basics = series_basics[series_basics['titleType'] == 'tvSeries'].copy()
series_basics = series_basics.dropna(subset=['startYear'])
series_basics['startYear'] = series_basics['startYear'].astype(int)


## Construção das métricas por série

Para cada série (`parentTconst`), calculamos:
- `numSeasons`: número de temporadas (máximo de `seasonNumber`)
- `avgEpisodesPerSeason`: média de episódios por temporada


In [ ]:
# número de temporadas por série
num_seasons = (
    episodes.groupby('parentTconst')['seasonNumber']
    .max()
    .rename('numSeasons')
)

# episódios por temporada, depois a média entre as temporadas de cada série
episodes_per_season = (
    episodes.groupby(['parentTconst', 'seasonNumber'])['episodeNumber']
    .count()
    .rename('episodeCount')
    .reset_index()
)

avg_episodes_per_season = (
    episodes_per_season.groupby('parentTconst')['episodeCount']
    .mean()
    .rename('avgEpisodesPerSeason')
)


In [ ]:
series_metrics = (
    series_basics
    .rename(columns={'tconst': 'parentTconst'})
    .merge(num_seasons, on='parentTconst', how='inner')
    .merge(avg_episodes_per_season, on='parentTconst', how='inner')
    .merge(
        ratings.rename(columns={'tconst': 'parentTconst'}),
        on='parentTconst',
        how='inner'
    )
)

# remover ratings com poucos votos
series_metrics = series_metrics[series_metrics['numVotes'] >= 100]

series_metrics.head()


In [ ]:
before = series_metrics[series_metrics['startYear'] < STREAMING_YEAR]
after = series_metrics[series_metrics['startYear'] >= STREAMING_YEAR]

print(f"Séries antes de {STREAMING_YEAR}: {len(before)}")
print(f"Séries após {STREAMING_YEAR}    : {len(after)}")


## Distribuições: antes x depois do streaming


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

sns.kdeplot(before['numSeasons'], label='Antes de 2010', fill=True, ax=axes[0])
sns.kdeplot(after['numSeasons'], label='Após 2010', fill=True, ax=axes[0])
axes[0].set_title('Número de Temporadas')
axes[0].set_xlabel('numSeasons')
axes[0].legend()

sns.kdeplot(before['avgEpisodesPerSeason'], label='Antes de 2010', fill=True, ax=axes[1])
sns.kdeplot(after['avgEpisodesPerSeason'], label='Após 2010', fill=True, ax=axes[1])
axes[1].set_title('Episódios por Temporada (média)')
axes[1].set_xlabel('avgEpisodesPerSeason')
axes[1].legend()

plt.tight_layout()
plt.show()


## Testes de hipótese

H0: não há diferença na distribuição de `numSeasons` / `avgEpisodesPerSeason` entre séries de antes e depois de 2010.

Usamos Mann-Whitney U (referência) e teste de permutação (confirmação).


In [ ]:
def permutation_test_diff_means(a, b, n_perm=10_000, rng=RNG):
    """
    Teste de permutação para diferença de médias entre dois grupos.
    H0: não há diferença entre os grupos (rótulo é irrelevante).
    Retorna a diferença observada e o p-valor bicaudal.
    """
    a = np.asarray(a)
    b = np.asarray(b)

    observed_diff = a.mean() - b.mean()

    pooled = np.concatenate([a, b])
    n_a = len(a)

    perm_diffs = np.empty(n_perm)

    for i in range(n_perm):
        shuffled = rng.permutation(pooled)
        perm_diffs[i] = shuffled[:n_a].mean() - shuffled[n_a:].mean()

    p_value = np.mean(np.abs(perm_diffs) >= np.abs(observed_diff))

    return observed_diff, p_value


In [ ]:
def run_ab_test(col, before, after, label):
    a = before[col].dropna()
    b = after[col].dropna()

    u_stat, p_mw = mannwhitneyu(a, b, alternative='two-sided')
    diff_obs, p_perm = permutation_test_diff_means(a, b)

    print("="*60)
    print(f"{label} - ANTES x APÓS {STREAMING_YEAR}")
    print("="*60)
    print(f"\nMédia antes : {a.mean():.3f}")
    print(f"Média após  : {b.mean():.3f}")
    print(f"Diferença observada (antes - após): {diff_obs:.3f}")
    print(f"\nMann-Whitney        : p = {p_mw:.5f}")
    print(f"Permutação (10.000) : p = {p_perm:.5f}")

    if p_mw < 0.05:
        print(f"\nConclusão: Existe diferença estatisticamente significativa em {label}.")
    else:
        print(f"\nConclusão: Não foi encontrada diferença significativa em {label}.")
    print()


In [ ]:
run_ab_test('numSeasons', before, after, 'NÚMERO DE TEMPORADAS')
run_ab_test('avgEpisodesPerSeason', before, after, 'EPISÓDIOS POR TEMPORADA')


## Correlação com qualidade (averageRating)

H0: não há correlação entre `numSeasons` / `avgEpisodesPerSeason` e `averageRating` da série.

Usamos Spearman + teste de permutação para o p-valor.


In [ ]:
def permutation_test_correlation(x, y, n_perm=10_000, rng=RNG):
    """
    Teste de permutação para correlação de Spearman.
    H0: não há correlação entre x e y (a ordem de y é irrelevante).
    Retorna o rho observado e o p-valor bicaudal.
    """
    x = np.asarray(x)
    y = np.asarray(y)

    observed_rho, _ = spearmanr(x, y)

    perm_rhos = np.empty(n_perm)

    for i in range(n_perm):
        shuffled_y = rng.permutation(y)
        perm_rhos[i], _ = spearmanr(x, shuffled_y)

    p_value = np.mean(np.abs(perm_rhos) >= np.abs(observed_rho))

    return observed_rho, p_value


In [ ]:
PERM_SAMPLE_SIZE = 20_000

sample = series_metrics.sample(
    n=min(PERM_SAMPLE_SIZE, len(series_metrics)),
    random_state=42
)

for col, label in [
    ('numSeasons', 'NÚMERO DE TEMPORADAS'),
    ('avgEpisodesPerSeason', 'EPISÓDIOS POR TEMPORADA')
]:
    rho_full, p_scipy = spearmanr(series_metrics[col], series_metrics['averageRating'])
    rho_sample, p_perm = permutation_test_correlation(
        sample[col], sample['averageRating']
    )

    print("="*60)
    print(f"{label} x AVERAGE RATING")
    print("="*60)
    print(f"\nSpearman rho (dataset completo) = {rho_full:.4f}")
    print(f"p-valor (scipy, dataset completo) = {p_scipy:.5f}")
    print(f"Spearman rho (amostra) = {rho_sample:.4f}")
    print(f"p-valor (permutação, 10.000, amostra) = {p_perm:.5f}")

    if p_perm < 0.05:
        print(f"\nConclusão: Existe correlação estatisticamente significativa "
              f"entre {label.lower()} e a nota da série.")
    else:
        print(f"\nConclusão: Não foi encontrada correlação estatisticamente "
              f"significativa entre {label.lower()} e a nota da série.")
    print()


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].scatter(
    series_metrics['numSeasons'],
    series_metrics['averageRating'],
    alpha=0.1, s=5
)
axes[0].set_xlabel('numSeasons')
axes[0].set_ylabel('averageRating')
axes[0].set_title('Temporadas x Nota')

axes[1].scatter(
    series_metrics['avgEpisodesPerSeason'],
    series_metrics['averageRating'],
    alpha=0.1, s=5
)
axes[1].set_xlabel('avgEpisodesPerSeason')
axes[1].set_ylabel('averageRating')
axes[1].set_title('Episódios/Temporada x Nota')

plt.tight_layout()
plt.show()
